# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print("Fields include:")
pp = pprint.PrettyPrinter(indent=2)
if hasattr(metadata, 'keywords'):
    pp.pprint(metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may contain multiple `RecordSet`s. Listing found `RecordSet` `@id`s, and their included `field` `@id`s where possible.

In [ ]:
# List all record sets in the dataset schema
record_sets = list(dataset.record_set_ids)
print("Available Record Sets (@id):")
for rs_id in record_sets:
    print(f"- {rs_id}")
    # Print corresponding fields for this record set
    fields = dataset.field_ids(record_set=rs_id)
    print(f"  Fields in record set ({rs_id}):")
    for f in fields:
        print(f"    - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. You may repeat for all record sets.

In [ ]:
# Extract data from each record set
# Use record set @ids from the above cell (e.g., 'cr:RecordSet/0' or as listed)
# Here we demonstrate with ALL available record sets for full exploration.
dataframes = {}

for rs_id in dataset.record_set_ids:
    print(f"Processing records in record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns (@id): {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set '{rs_id}'.")
        
# Choose a record set to explore
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    print(f"Using record set for analysis: {record_set_id}")
    print("Available columns for EDA:")
    print(dataframes[record_set_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a numeric field for demonstration. You may need to change the field `@id` to match actual column names for your chosen record set, as described above.

In [ ]:
# Choose a numeric field @id. Example: 'cr:field/coverage_percent', adjust as needed.
ex_df = dataframes[record_set_id]
numeric_candidates = [c for c in ex_df.columns if ex_df[c].dtype in [float, int] or pd.api.types.is_numeric_dtype(ex_df[c])]
print("Numeric fields detected:", numeric_candidates)

# If no numeric field detected, try fallback by inferring numeric values in string columns
if not numeric_candidates:
    for col in ex_df.columns:
        try:
            ex_df[col] = pd.to_numeric(ex_df[col])
            if pd.api.types.is_numeric_dtype(ex_df[col]):
                numeric_candidates.append(col)
        except Exception:
            continue
    print("Numeric fields after conversion:", numeric_candidates)

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for demo: {numeric_field}")
 
    threshold = ex_df[numeric_field].mean()
    filtered_df = ex_df[ex_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    normcol = f"{numeric_field}_normalized"
    filtered_df[normcol] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normcol]].head())

    # Try grouping by a non-numeric field
    group_candidates = [c for c in ex_df.columns if c != numeric_field and ex_df[c].dtype == object]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(ex_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if len(numeric_candidates) > 1:
        # Plot scatter between 2 numeric fields
        second_numeric = numeric_candidates[1]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(data=ex_df, x=numeric_field, y=second_numeric)
        plt.xlabel(numeric_field)
        plt.ylabel(second_numeric)
        plt.title(f"{numeric_field} vs {second_numeric}")
        plt.show()
else:
    print("No numeric fields to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the FAIR^2 mass spectrometry dataset via Croissant schema using `mlcroissant`.
- Explored available record sets and fields, with columns referenced by their unique `@id` identifiers.
- Extracted and analyzed data in pandas DataFrames, with filtering, normalization, and simple grouping by field attributes.
- Visualized numeric field distributions and demonstrated further EDA pathways.

You may adapt the sections above for your specific downstream analysis, feature selection, or modeling needs.